<a href="https://colab.research.google.com/github/Faizan-Rashid/deep-learning-pytorch/blob/main/05_pytorch_going_modular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **05. Pytorch Going Modular**

### What is going modular?

Going modular involves turning notebook code (from a Jupyter Notebook or Google Colab notebook) into a series of different Python scripts that offer similar functionality.

For example, we could turn our notebook code from a series of cells into the following Python files:

- `data_setup.py` - a file to prepare and download data if needed.
- `engine.py` - a file containing various training functions.
- `model_builder.py` or model.py - a file to create a PyTorch model.
- `train.py` - a file to leverage all other files and train a target PyTorch model.
- `utils.py` - a file dedicated to helpful utility functions.

### Why would you want to go modular?

Notebooks are fantastic for iteratively exploring and running experiments quickly.

However, for larger scale projects you may find Python scripts more reproducible and easier to run.

Though this is a debated topic, as companies like Netflix have shown how they use notebooks for production code.

Production code is code that runs to offer a service to someone or something.

For example, if you have an app running online that other people can access and use, the code running that app is considered production code.

And libraries like `fast.ai's` `nb-dev` (short for notebook development) enable you to write whole Python libraries (including documentation) with Jupyter Notebooks.

### What we're going to cover

The main concept of this section is: **turn useful notebook code cells into reusable Python files**.

### The Goal of This Notebook

1. he ability to train the model we built in notebook 04 (Food Vision Mini) with one line of code on the command line: python train.py.
2. A directory structure of reusable Python scripts, such as:

```going_modular/
├── going_modular/
│   ├── data_setup.py
│   ├── engine.py
│   ├── model_builder.py
│   ├── train.py
│   └── utils.py
├── models/
│   ├── 05_going_modular_cell_mode_tinyvgg_model.pth
│   └── 05_going_modular_script_mode_tinyvgg_model.pth
└── data/
    └── pizza_steak_sushi/
        ├── train/
        │   ├── pizza/
        │   │   ├── image01.jpeg
        │   │   └── ...
        │   ├── steak/
        │   └── sushi/
        └── test/
            ├── pizza/
            ├── steak/
            └── sushi/
```

## 0. Cell mode Vs Script mode

A cell mode notebook is a notebook run normally, each cell in the notebook is either code or markdown.

A script mode notebook is very similar to a cell mode notebook, however, many of the code cells may be turned into Python scripts.

***Note:** You don't need to create Python scripts via a notebook, you can create them directly through an IDE (integrated developer environment) such as VS Code. Having the script mode notebook as part of this section is just to demonstrate one way of going from notebooks to Python scripts.*

## 1. Get data

Same as notebook 04

In [ ]:
import torch
import os
import requests
from pathlib import Path
import zipfile

In [ ]:
# Device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [ ]:
data_path = Path("data")

image_path = data_path / "pizza_steak_sushi"

# If the image folder doesn't exist, download it and prepare it...
if image_path.is_dir():
    print(f"{image_path} directory exists.")
else:
    print(f"Did not find {image_path} directory, creating one...")
    Path.mkdir(self=image_path, exist_ok=True, parents=True)

# If zip fie not present here download from github
if not (data_path / "pizza_steak_sushi.zip").is_file():
  # get the zip file
  r = requests.get("https://github.com/Faizan-Rashid/deep-learning-pytorch/raw/refs/heads/main/extras/data/pizza_steak_sushi.zip")
  # if not present here then make one with this name and write to it
  with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
    f.write(r.content)

# unzip file in image_path folder
with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
  zip_ref.extractall(path=image_path)

# Remove zip file
os.remove(data_path / "pizza_steak_sushi.zip")

Did not find data/pizza_steak_sushi directory, creating one...


## 2. Create Datasets and DataLoaders `(data_setup.py)`

In [ ]:
import os
os.makedirs("going_modular")

In [ ]:
%%writefile going_modular/data_setup.py

"""Has functionality for creating train and test dataloaders"""

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

def create_dataloaders(transform: transforms.Compose,
                       train_path: str,
                       test_path: str,
                       BATCH_SIZE: int,
                       NUM_WORKERS: int = os.cpu_count()):

  # Create dataset object from path
  train_data = datasets.ImageFolder(train_path, transform)
  test_data = datasets.ImageFolder(test_path, transform)

  # Classnames
  class_names = train_data.classes

  # DataLoaders
  train_dataloader = DataLoader(dataset=train_data,
                                batch_size=BATCH_SIZE,
                                shuffle=True,
                                num_workers=NUM_WORKERS)

  test_dataloader = DataLoader(dataset=test_data,
                               batch_size=BATCH_SIZE,
                               shuffle=False,
                               num_workers=NUM_WORKERS)

  return train_dataloader, test_dataloader, class_names

Writing going_modular/data_setup.py


## 3. Making Model (`model_builder.py`)



In [ ]:
%%writefile going_modular/model_builder.py

import torch
import torch.nn as nn

class TinyVGG(nn.Module):
  def __init__(self, input_shape: int,
               hidden_units: int = 10,
               output_shape: int = 3):

    super().__init__()

    self.block1 = nn.Sequential(
        nn.Conv2d(input_shape, hidden_units, 3, 1, 1),
        nn.ReLU(),

        nn.Conv2d(hidden_units, hidden_units, 3, 1, 1),
        nn.ReLU(),
        nn.MaxPool2d(3, 2)
    )

    self.block2 = nn.Sequential(
        nn.Conv2d(hidden_units, hidden_units, 3, 1),
        nn.ReLU(),

        nn.Conv2d(hidden_units, hidden_units, 3, 1),
        nn.MaxPool2d(2, 2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(1690, output_shape)
    )

  def forward(self, x):
    return self.classifier(self.block2(self.block1(x)))

Overwriting going_modular/model_builder.py


In [ ]:
# e.g to use script
from going_modular import model_builder

torch.manual_seed(42)
model = model_builder.TinyVGG(3, 10, 3).to(device)

## 4. 4. Creating train_step() and test_step() functions and train() to combine them

In [ ]:
%%writefile going_modular/engine.py

import torch
import torch.nn as nn

from typing import Callable
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

# Train Step Function
def train_step(model: nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device=device):
  model.train()

  train_loss, acc = 0, 0
  for batch, (X, y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)
    # Forward pass
    pred = model(X)

    # Calculate loss
    loss = loss_fn(pred, y)
    train_loss += loss.item()
    acc += (torch.softmax(pred, dim=1).argmax(dim=1) == y).sum().item() / len(pred)

    # zero gradient
    optimizer.zero_grad()

    # back propogation
    loss.backward()

    # optimizer step
    optimizer.step()

  train_loss /= len(dataloader)
  acc /= len(dataloader)

  print(f"train loss : {train_loss}")
  print(f"train accuracy : {acc}")

  return train_loss, acc


# Test Step Function
def test_step(model: nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: nn.Module,
              device: torch.device=device):
  model.eval()
  test_loss, acc = 0, 0
  with torch.inference_mode():
    for batch, (X, y) in enumerate(dataloader):
      X, y = X.to(device), y.to(device)
      pred = model(X)
      test_loss += loss_fn(pred, y).item()
      acc += (torch.softmax(pred, dim=1).argmax(dim=1) == y).sum().item() / len(pred)

    test_loss /= len(dataloader)
    acc /= len(dataloader)

    print(f"test loss : {test_loss}")
    print(f"test accuracy : {acc}")

  return test_loss, acc


# Train function
def train(model: nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: nn.Module=nn.CrossEntropyLoss(),
          epochs: int=5,
          device: torch.device=device):

  results = {
      "train_loss": [],
      "test_loss": [],
      "train_acc": [],
      "test_acc": []
  }

  for epoch in tqdm(range(epochs)):
    print(f"epoch : {epoch}\n----------")

    # do training - call train step
    train_loss, train_acc = train_step(model=model,
                                      dataloader=train_dataloader,
                                      loss_fn=loss_fn,
                                      optimizer=optimizer,
                                      device=device)

    # do testing - call test step
    test_loss, test_acc = test_step(model=model,
                                    dataloader=test_dataloader,
                                    loss_fn=loss_fn,
                                    device=device)

    # store values in results
    results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
    results["test_loss"].append(test_loss.item() if isinstance(test_loss, torch.Tensor) else test_loss)
    results["train_acc"].append(train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc)
    results["test_acc"].append(test_acc.item() if isinstance(test_acc, torch.Tensor) else test_acc)

  # return results
  return results

Overwriting going_modular/engine.py


In [ ]:
from going_modular import engine

# engine.train(....)

## 5. Creating a function to save the model (utils.py)

In [ ]:
%%writefile going_modular/utils.py

import torch.nn as nn
from pathlib import Path

def save_model(model: nn.Module,
               target_dir: str,
               model_name: str):

  target_dir_path = Path(target_dir)
  target_dir_path.mkdir(parents=True,
                        exist_ok=True)

  assert model_name.endswith("pt") or model_name.endswith(".pth") , "model_name should end with .pt ot .pth"

  model_save_path = target_dir_path / model_name

  torch.save(obj = model.state_dict(),
             f=model_save_path)


Overwriting going_modular/utils.py


In [ ]:
from going_modular import utils

# utils.save_model(...)

## 6. Train, evaluate and save the model (`train.py`)

In [ ]:
%%writefile going_modular/train.py

import os
import torch
import data_setup, engine, model_builder, utils
from torchvision import transforms

# Hyper-parameters
NUM_WORKERS = os.cpu_count()
BATCH_SIZE = 32
NUM_EPOCHS = 5
HIDDEN_UNITS = 10
LEARNING_RATE = 0.001

# train and test directories
train_dir = "data/pizza_steak_sushi/train"
test_dir = "data/pizza_steak_sushi/test"

# device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"

# transforms
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# create dataloaders with the help of data_setup
train_dataloader, test_dataloader, classes = data_setup.create_dataloaders(transform=transform,
                              train_path=train_dir,
                              test_path=test_dir,
                              BATCH_SIZE=BATCH_SIZE,
                              NUM_WORKERS=NUM_WORKERS)

# create model by model_builder.py
model = model_builder.TinyVGG(input_shape=3,
                              hidden_units=HIDDEN_UNITS,
                              output_shape=len(classes))

# loss function and optimzer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(),
                             lr=LEARNING_RATE)

# train the model by using train from engine.py
train(model=model,
      train_dataloader=train_dataloader,
      test_dataloader=test_dataloader,
      optimizer=optimizer,
      loss_fn=loss_fn,
      epochs=NUM_EPOCHS,
      device=device)

# save model using utils.py
utils.save_model(model=model,
                 target_dir="model",
                 model_name="05_going_modular_script_TinyVGG.pth")

Overwriting going_modular/train.py


In [ ]:
train

<function __main__.train(model: torch.nn.modules.module.Module, train_dataloader: torch.utils.data.dataloader.DataLoader, test_dataloader: torch.utils.data.dataloader.DataLoader, optimizer: torch.optim.optimizer.Optimizer, loss_fn: torch.nn.modules.module.Module, epochs: int, device: torch.device) -> Dict[str, List]>